In [1]:
import os
from pathlib import Path
import json
from pprint import pprint

from torch import nn as nn
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline, \
    DataCollatorForTokenClassification, Trainer
from datasets import load_from_disk

from utils import get_best_checkpoint,compute_metrics, tokenize_and_align_labels


In [2]:
DATA_DIR = Path(os.getcwd()).parent / "data"

LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"

MODELS_DIR = Path(os.getcwd()).parent/"models"
MODELS_DIR.mkdir(exist_ok=True)
dataset = load_from_disk(DATASET_PATH) # type:ignore
# label info
with open(LABEL_INFO_PATH) as f:
    label_info: dict = json.load(f)
label2id: dict[str, int] = label_info["label2id"]
id2label = label_info["id2label"]

In [3]:
# Registry: maps each variant to its HF repo and human-readable metadata
MODELS = [
    {
        "dir_name":    "small-fully_labeled",
        "repo_id":     None,
        "model_name":  None,
        "base_model":  "microsoft/deberta-v3-small",
        "size_label":  "Base (44M params)",
        "macro_f1":    0.9299,
        "inference_speed": None,
    },
    {
        "dir_name":    "base-bf16-weighted-trainer",
        "repo_id":     "bengid/pii-redaction-deberta-base",
        "model_name":  "DeBERTa-v3-Base PII Redaction",
        "base_model":  "microsoft/deberta-v3-base",
        "size_label":  "Base (86M params)",
        "macro_f1":    0.9557,
        "inference_speed": "~11.7ms on RTX 5070",
    },
    {
        "dir_name":    "small-bf16-weighted-trainer",
        "repo_id":     "bengid/pii-redaction-deberta-small",
        "model_name":  "DeBERTa-v3-Small PII Redaction",
        "base_model":  "microsoft/deberta-v3-small",
        "size_label":  "Small (44M params)",
        "macro_f1":    0.9476,
        "inference_speed": "~6.5ms on RTX 5070",
    },
    {
        "dir_name":    "xsmall-bf16-weighted-trainer",
        "repo_id":     "bengid/pii-redaction-deberta-xsmall",
        "model_name":  "DeBERTa-v3-XSmall PII Redaction",
        "base_model":  "microsoft/deberta-v3-xsmall",
        "size_label":  "XSmall (22M params)",
        "macro_f1":    0.9303,
        "inference_speed": "~11.6ms on RTX 5070 [1]"
    }
]

In [4]:
dataset = dataset.map(tokenize_and_align_labels, batched=True, batch_size=20_000)


for model_cfg in MODELS:
    model_dir = MODELS_DIR / model_cfg["dir_name"]
    best_ckpt, _, _ = get_best_checkpoint(model_dir)
    
    model= AutoModelForTokenClassification.from_pretrained(best_ckpt)
    tokenizer = AutoTokenizer.from_pretrained(best_ckpt)
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    trainer = Trainer(
        model=model, 
        compute_metrics=compute_metrics,
        processing_class=tokenizer,
        data_collator=data_collator
    )
    
    test_results = trainer.predict(dataset["test"])

    print(test_results.metrics)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'test_loss': 0.07009364664554596, 'test_model_preparation_time': 0.0011, 'test_precision': 0.9395035991031743, 'test_recall': 0.9525423728813559, 'test_f1': 0.9459780585369717, 'test_accuracy': 0.9857750092066088, 'test_BOD_f1': 0.9685314685314687, 'test_BOD_support': 1142, 'test_BUILDING_f1': 0.9872881355932203, 'test_BUILDING_support': 943, 'test_CITY_f1': 0.9681908548707754, 'test_CITY_support': 996, 'test_COUNTRY_f1': 0.9693941286695815, 'test_COUNTRY_support': 790, 'test_DATE_f1': 0.9206896551724139, 'test_DATE_support': 862, 'test_DRIVERLICENSE_f1': 0.9356872635561161, 'test_DRIVERLICENSE_support': 1192, 'test_EMAIL_f1': 0.9871645274212368, 'test_EMAIL_support': 1279, 'test_GEOCOORD_f1': 0.9910714285714286, 'test_GEOCOORD_support': 112, 'test_GIVENNAME1_f1': 0.8594415522953147, 'test_GIVENNAME1_support': 1026, 'test_GIVENNAME2_f1': 0.7962962962962963, 'test_GIVENNAME2_support': 270, 'test_IDCARD_f1': 0.9247058823529412, 'test_IDCARD_support': 1251, 'test_IP_f1': 0.97910990009082

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

{'test_loss': 0.05264482647180557, 'test_model_preparation_time': 0.0015, 'test_precision': 0.9525278898647045, 'test_recall': 0.9602392821535394, 'test_f1': 0.9563680416261197, 'test_accuracy': 0.9911179631860484, 'test_BOD_f1': 0.9764192139737993, 'test_BOD_support': 1142, 'test_BUILDING_f1': 0.9882978723404255, 'test_BUILDING_support': 943, 'test_CITY_f1': 0.9789368104312939, 'test_CITY_support': 996, 'test_COUNTRY_f1': 0.9606003752345216, 'test_COUNTRY_support': 790, 'test_DATE_f1': 0.9337938975244674, 'test_DATE_support': 862, 'test_DRIVERLICENSE_f1': 0.9509844993715961, 'test_DRIVERLICENSE_support': 1192, 'test_EMAIL_f1': 0.9890625, 'test_EMAIL_support': 1279, 'test_GEOCOORD_f1': 0.9779735682819383, 'test_GEOCOORD_support': 112, 'test_GIVENNAME1_f1': 0.8895348837209303, 'test_GIVENNAME1_support': 1026, 'test_GIVENNAME2_f1': 0.8117647058823529, 'test_GIVENNAME2_support': 270, 'test_IDCARD_f1': 0.9411293552262716, 'test_IDCARD_support': 1251, 'test_IP_f1': 0.992255125284738, 'test_

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'test_loss': 0.051957983523607254, 'test_model_preparation_time': 0.0009, 'test_precision': 0.9451335124032233, 'test_recall': 0.9542173479561316, 'test_f1': 0.9496537080034133, 'test_accuracy': 0.9905343578156306, 'test_BOD_f1': 0.9741341516878562, 'test_BOD_support': 1142, 'test_BUILDING_f1': 0.9861702127659574, 'test_BUILDING_support': 943, 'test_CITY_f1': 0.9681592039800995, 'test_CITY_support': 996, 'test_COUNTRY_f1': 0.9617074701820464, 'test_COUNTRY_support': 790, 'test_DATE_f1': 0.9255441008018329, 'test_DATE_support': 862, 'test_DRIVERLICENSE_f1': 0.9384679782335705, 'test_DRIVERLICENSE_support': 1192, 'test_EMAIL_f1': 0.9914263445050662, 'test_EMAIL_support': 1279, 'test_GEOCOORD_f1': 0.9691629955947136, 'test_GEOCOORD_support': 112, 'test_GIVENNAME1_f1': 0.8670465337132003, 'test_GIVENNAME1_support': 1026, 'test_GIVENNAME2_f1': 0.8127340823970037, 'test_GIVENNAME2_support': 270, 'test_IDCARD_f1': 0.9349206349206349, 'test_IDCARD_support': 1251, 'test_IP_f1': 0.9890610756608

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

{'test_loss': 0.04940490797162056, 'test_model_preparation_time': 0.0017, 'test_precision': 0.9368765524583054, 'test_recall': 0.9476370887337986, 'test_f1': 0.942226099369523, 'test_accuracy': 0.9895762463251587, 'test_BOD_f1': 0.9710526315789474, 'test_BOD_support': 1142, 'test_BUILDING_f1': 0.9841101694915254, 'test_BUILDING_support': 943, 'test_CITY_f1': 0.963771712158809, 'test_CITY_support': 996, 'test_COUNTRY_f1': 0.9620253164556962, 'test_COUNTRY_support': 790, 'test_DATE_f1': 0.9213872832369943, 'test_DATE_support': 862, 'test_DRIVERLICENSE_f1': 0.92399658411614, 'test_DRIVERLICENSE_support': 1192, 'test_EMAIL_f1': 0.9871244635193133, 'test_EMAIL_support': 1279, 'test_GEOCOORD_f1': 0.9866666666666667, 'test_GEOCOORD_support': 112, 'test_GIVENNAME1_f1': 0.8476506881822496, 'test_GIVENNAME1_support': 1026, 'test_GIVENNAME2_f1': 0.7752808988764045, 'test_GIVENNAME2_support': 270, 'test_IDCARD_f1': 0.9216589861751152, 'test_IDCARD_support': 1251, 'test_IP_f1': 0.990424076607387, '

In [5]:
for model_cfg in MODELS:
    model_dir = MODELS_DIR / model_cfg["dir_name"]
    
    # Get best checkpoint from trainer state
    best_ckpt, _, _ = get_best_checkpoint(model_dir)
    
    
    model= AutoModelForTokenClassification.from_pretrained(best_ckpt)
    params = sum(
        p.numel() for name, p in model.named_parameters()
        if "embedding" not in name.lower()
    )
        
    layers = model.config.num_hidden_layers
    
    print(f"{model_cfg['repo_id']}, {params:,}, {layers=}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

None, 42,571,063, layers=6


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

bengid/pii-redaction-deberta-base, 85,098,295, layers=12


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

bengid/pii-redaction-deberta-small, 42,571,063, layers=6


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

bengid/pii-redaction-deberta-xsmall, 21,315,511, layers=12


In [9]:
text = dataset["train"][1]["source_text"]
text2 = "my name is Benyamin and i live at 1234 s holt street. my phone number is +1 (124) 567-3434"
text3 = "My name is John Smith and my email is john@example.com"
text4 = "She lives at 742 Evergreen Terrace, Springfield, IL 62704."

out = {}

for model_cfg in MODELS:
    model_dir = MODELS_DIR / model_cfg["dir_name"]
    best_ckpt, _, _ = get_best_checkpoint(model_dir)
    
    
    model= AutoModelForTokenClassification.from_pretrained(best_ckpt)
    tokenizer = AutoTokenizer.from_pretrained(best_ckpt)
    
    pipe = pipeline(
        task="token-classification",
        model=model,
        aggregation_strategy="first",
        tokenizer=tokenizer
    )
    entities = pipe(text)
    entities2 = pipe(text2)
    entities3 = pipe(text3)
    entities4 = pipe(text4)
    out[model_cfg["model_name"]] = [entities, entities2, entities3, entities4]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/home/zelluzy/Desktop/pii_redaction_api/.venv/lib/python3.14/site-packages/transformers/pipelines/token_classification.py:444: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

In [12]:
curr_model = 2
curr_text = 3
pprint(out[MODELS[curr_model]["model_name"]][curr_text])

[{'end': 16,
  'entity_group': 'BUILDING',
  'score': np.float32(0.998517),
  'start': 12,
  'word': '742'},
 {'end': 35,
  'entity_group': 'STREET',
  'score': np.float32(0.99809444),
  'start': 16,
  'word': 'EvergreenTerrace,'},
 {'end': 48,
  'entity_group': 'CITY',
  'score': np.float32(0.9992367),
  'start': 35,
  'word': 'Springfield,'},
 {'end': 51,
  'entity_group': 'STATE',
  'score': np.float32(0.9984706),
  'start': 48,
  'word': 'IL'},
 {'end': 58,
  'entity_group': 'POSTCODE',
  'score': np.float32(0.97415483),
  'start': 51,
  'word': '62704.'}]


In [8]:
enc = tokenizer(text2, return_offsets_mapping=True)
for tok, off, wid in zip(enc.tokens(), enc["offset_mapping"], enc.word_ids()):
    print(f"{tok:20s} {off} {wid}  →  '{text2[off[0]:off[1]]}'")

[CLS]                (0, 0) None  →  ''
▁my                  (0, 2) 0  →  'my'
▁name                (2, 7) 1  →  ' name'
▁is                  (7, 10) 2  →  ' is'
▁Ben                 (10, 14) 3  →  ' Ben'
yam                  (14, 17) 3  →  'yam'
in                   (17, 19) 3  →  'in'
▁and                 (19, 23) 4  →  ' and'
▁i                   (23, 25) 5  →  ' i'
▁live                (25, 30) 6  →  ' live'
▁at                  (30, 33) 7  →  ' at'
▁12                  (33, 36) 8  →  ' 12'
34                   (36, 38) 8  →  '34'
▁s                   (38, 40) 9  →  ' s'
▁hol                 (40, 44) 10  →  ' hol'
t                    (44, 45) 10  →  't'
▁street              (45, 52) 11  →  ' street'
.                    (52, 53) 11  →  '.'
▁my                  (53, 56) 12  →  ' my'
▁phone               (56, 62) 13  →  ' phone'
▁number              (62, 69) 14  →  ' number'
▁is                  (69, 72) 15  →  ' is'
▁+                   (72, 74) 16  →  ' +'
1                    (74